<a href="https://colab.research.google.com/github/dhag/DuplexChannel/blob/main/duplexchannel_ws_binary_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DuplexChannel — WebSocket バイナリ送受信サンプル（Google Colab）

`dhag/DuplexChannel` の Python ライブラリ（`Other/haglib_duplex.py`）を GitHub から取得し、
**同一セッション内でサーバーとクライアントを起動**して、**生バイナリフレーム（DuplexPacket）**で
双方向通信する最小サンプルです。

- 送信フレーム種別は `FrameKind.BINARY`（`DuplexPacket` をそのままバイナリ WebSocket フレームで送出）
- 実演内容: プッシュ送信 / リクエスト・レスポンス / ブロードキャスト

> メモ: 上のメニュー「ランタイム → すべてのセルを実行」で順に実行してください。


## 1. 依存インストール & ライブラリ取得（GitHub から clone）

In [1]:
# websockets を導入し、ライブラリを GitHub から取得
!pip -q install websockets

import os
if not os.path.exists("DuplexChannel"):
    !git clone --depth 1 https://github.com/dhag/DuplexChannel.git
else:
    print("already cloned")


Cloning into 'DuplexChannel'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 107 (delta 22), reused 62 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 570.89 KiB | 11.19 MiB/s, done.
Resolving deltas: 100% (22/22), done.


## 2. import（`Other/` を import パスに追加）

https://github.com/dhag/DuplexChannel のOther/haglib_duplex.pyに、関数群（ライブラリ化してない生の関数群）が入っている。

In [2]:
import sys
sys.path.insert(0, "DuplexChannel/Other")

import haglib_duplex as H

# 生バイナリ対応（FrameKind）が入っていることを確認
print("FrameKind:", list(H.FrameKind))
print("has send_bytes_frame:", hasattr(H.WebSocketDuplexChannel, "send_bytes_frame"))


FrameKind: [<FrameKind.TEXT: 0>, <FrameKind.BINARY: 1>]
has send_bytes_frame: True


## 3. デモ本体

同一ノートブック内で **サーバー** と **クライアント** を `default_frame=FrameKind.BINARY` で起動します。
Colab（IPython カーネル）はセル内でのトップレベル `await` に対応しているため、`await demo()` で実行します。


In [3]:
import asyncio

def hx(b: bytes) -> str:
    return b.hex(" ") if b else "(empty)"

async def demo():
    PORT = 8781

    # ---- サーバー（既定フレーム = BINARY）----
    server = H.WebSocketDuplexServer(default_frame=H.FrameKind.BINARY)
    #クライアントから受信したときに飛んでくる関数を作っておく
    async def on_server_received(ch, msg):
        tp = msg.to_typed_payload()
        data = tp.get_binary()
        if msg.type == H.MessageType.REQUEST:
            print(f"[server] request  : {hx(data)}  -> reply PONG:")
            reply = H.TypedPayload.from_binary(b"PONG:" + (data or b"")).to_message()
            await ch.reply(msg, reply)            # kind 省略 = サーバーの既定 BINARY
        else:
            print(f"[server] push recv : {hx(data)}")
    #クライアントから受信したときにその関数が呼び出されるように設定しておく。
    server.on_received = on_server_received
    #サーバを起動する。
    await server.start(PORT, "localhost")
    print(f"[server] listening on ws://localhost:{PORT}/")

    # ---- クライアント（既定フレーム = BINARY）----
    client = H.WebSocketDuplexClient(f"ws://localhost:{PORT}", default_frame=H.FrameKind.BINARY)

    #サーバーから受信したときに飛んでくる関数を作っておく
    async def on_client_received(ch, msg):
        print(f"[client] broadcast: {hx(msg.to_typed_payload().get_binary())}")

    #サーバーから受信したときにその関数が呼び出されるように設定しておく。
    client.on_received = on_client_received
    #サーバに接続する
    await client.connect()
    await asyncio.sleep(0.1)

    # (1) クライアントからサーバへ生バイナリをプッシュ送信してみる（client -> server）
    await client.send_bytes_frame(bytes([0x01, 0x02, 0x03]))
    await asyncio.sleep(0.1)

    # (2) クライアントからサーバへリクエスト/サーバのレスポンスを受けてみる（binary 往復、id で相関）
    resp = await client.send_and_receive(
        H.TypedPayload.from_binary(b"PING").to_message()
    )
    print(f"[client] response : {hx(resp.to_typed_payload().get_binary())}")

    # (3) サーバーから全クライアントへ生バイナリをブロードキャストする。
    await server.broadcast_bytes_frame(bytes([0xAA, 0xBB]))
    await asyncio.sleep(0.1)

    # ---- 後片付け ----
    await client.close()
    await server.stop()
    print("done.")

await demo()


[server] listening on ws://localhost:8781/
[server] push recv : 01 02 03
[server] request  : 50 49 4e 47  -> reply PONG:
[client] response : 50 4f 4e 47 3a 50 49 4e 47
[client] broadcast: aa bb
done.


## ポイント

- `default_frame=FrameKind.BINARY` にすると、`send` / `send_and_receive` / `reply` / `broadcast`
  の **kind 省略呼び出しがすべてバイナリフレーム**（`DuplexPacket`）で送出されます。
- 個別に切り替えたい場合は各メソッドに `kind=FrameKind.TEXT` / `FrameKind.BINARY` を渡します
  （例: `await client.send(msg, kind=H.FrameKind.TEXT)`）。
- 生の `bytes` を送るヘルパー `send_bytes_frame(data)` / `broadcast_bytes_frame(data)` は、
  受信側の `to_typed_payload()` が成立するよう内部で `TypedPayload(Binary)` に包みます。
- `TypedPayload` を明示して作る場合は
  `H.TypedPayload.from_binary(b"...").to_message()` のように渡します。
- ワイヤ形式（`DuplexPacket` 16 バイトヘッダ + `TypedPayload`）は C# 実装と同一なので、
  この Python をそのまま C# の `WebSocketDuplexServer` / `WebSocketDuplexClient` と相互接続できます。
